# **RAG (Retrieval Augmented Generation) 튜토리얼**

## **1. 실습환경 구성**

**1-1. 패키지 설치**

In [ ]:
!pip install langchain langchain-openai langchain-community faiss-cpu pypdf tiktoken beautifulsoup4 -q

**1-2. API 키 설정**

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = "YOUR API KEY"


## **2. RAG 개요**

**RAG(Retrieval Augmented Generation)** 는 LLM의 한계를 보완하기 위한 기술입니다.

```
사용자 질문
     │
     ▼
[Retriever] ──▶ 벡터 스토어(Vector Store) 검색
     │               ↑ 임베딩(Embedding)
     │           문서(Documents)
     ▼
[관련 문서 + 질문] ──▶ LLM ──▶ 최종 답변
```

**왜 RAG가 필요한가?**
- LLM은 학습 데이터 이후의 정보를 모름 (knowledge cutoff)
- 사내 문서, 전용 지식 등 private 데이터에 접근 불가
- Hallucination(환각) 발생 가능성

**RAG의 핵심 구성요소:**
1. **Document Loader** – PDF, 웹, DB 등에서 문서 로딩
2. **Text Splitter** – 긴 문서를 청크 단위로 분할
3. **Embedding Model** – 텍스트 → 벡터 변환
4. **Vector Store** – 벡터 저장 및 유사도 검색
5. **Retriever** – 쿼리와 유사한 문서 검색
6. **LLM** – 검색된 문서 기반 답변 생성


## **3. 문서 로딩 (Document Loading)**

**3-1. 텍스트 직접 입력 (Document 객체)**

In [ ]:
from langchain_core.documents import Document

# 샘플 문서 생성 (실제 업무에서는 PDF, DB, 웹 등에서 로딩)
documents = [
    Document(
        page_content="""이노코어(Innocore)는 AI 기반 솔루션을 개발하는 스타트업입니다.
2020년에 설립되어 제조업, 물류, 헬스케어 분야에 특화된 AI 서비스를 제공합니다.
주요 제품으로는 PRISM-AI 플랫폼이 있으며, 이를 통해 비정형 데이터 분석과
의사결정 지원 시스템을 구축할 수 있습니다. 현재 국내 50개 이상 기업에 서비스 중입니다.""",
        metadata={"source": "company_overview", "page": 1}
    ),
    Document(
        page_content="""PRISM-AI는 이노코어의 핵심 AI 플랫폼으로, 자연어 처리(NLP),
컴퓨터 비전, 예측 분석 기능을 통합한 솔루션입니다.
산업 현장의 데이터를 실시간으로 수집하고 분석하여 이상 감지,
품질 관리, 수요 예측 등의 서비스를 제공합니다.
2025년 기준 처리 데이터 규모는 일 10TB 이상입니다.""",
        metadata={"source": "product_info", "page": 1}
    ),
    Document(
        page_content="""RAG(Retrieval Augmented Generation)의 주요 구성 요소:
1. 문서 로더(Document Loader): PDF, 웹페이지, 데이터베이스 등에서 문서를 로딩
2. 텍스트 분할기(Text Splitter): 긴 문서를 청크 단위로 분할
3. 임베딩 모델(Embedding Model): 텍스트를 벡터로 변환
4. 벡터 스토어(Vector Store): 벡터를 저장하고 유사도 검색 수행
5. 검색기(Retriever): 쿼리와 의미적으로 유사한 문서 검색
6. LLM: 검색된 문서를 바탕으로 최종 답변 생성""",
        metadata={"source": "rag_overview", "page": 1}
    ),
    Document(
        page_content="""LLM(Large Language Model)은 대규모 텍스트 데이터로 학습된 언어 모델입니다.
GPT-4o, Claude 3.5 Sonnet, Gemini 1.5 Pro 등이 대표적입니다.
LLM의 한계: 학습 데이터 이후 최신 정보 부재, private 데이터 접근 불가,
긴 문서 처리의 비효율, 높은 비용 등이 있습니다.
RAG는 이러한 한계를 보완하는 실용적인 아키텍처입니다.""",
        metadata={"source": "llm_concepts", "page": 1}
    ),
    Document(
        page_content="""이노코어 워크숍 2026 커리큘럼:
Day 1: LangChain 기초 - LLM 체인, 프롬프트 엔지니어링, 메모리, 챗봇 구현
Day 2: RAG 구현 - 문서 로딩, 벡터 스토어, RAG 체인, 대화형 RAG
Day 3: Multi-agent 시스템 - LangGraph 그래프 설계, AutoGen 팀 구성
워크숍 목표: 실무에서 바로 활용 가능한 LLM 기반 AI 시스템 구축 역량 강화""",
        metadata={"source": "workshop_info", "page": 1}
    ),
]

print(f"총 {len(documents)}개 문서 로딩 완료")
for doc in documents:
    print(f"  - [{doc.metadata['source']}] {doc.page_content[:50]}...")


**3-2. PDF 문서 로딩**

In [ ]:
# Colab에서 PDF 파일 업로드 후 사용하는 방법
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader

# 방법 1: 파일 업로드
# uploaded = files.upload()
# pdf_loader = PyPDFLoader(list(uploaded.keys())[0])
# pdf_docs = pdf_loader.load()

# 방법 2: URL에서 직접 로딩 (공개 PDF)
import urllib.request
pdf_url = "https://arxiv.org/pdf/2005.11401"  # RAG 원논문
urllib.request.urlretrieve(pdf_url, "/tmp/rag_paper.pdf")

pdf_loader = PyPDFLoader("/tmp/rag_paper.pdf")
pdf_docs = pdf_loader.load()

print(f"PDF 로딩 완료: {len(pdf_docs)}페이지")
print(f"첫 번째 페이지 미리보기:\n{pdf_docs[0].page_content[:300]}...")


**3-3. 웹 페이지 로딩**

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
import bs4

# 웹 페이지에서 특정 태그만 추출
loader = WebBaseLoader(
    web_paths=["https://n.news.naver.com/article/030/0003268099"],
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(["h1", "h2", "p"])  # 제목, 본문만 추출
    )
)
web_docs = loader.load()
print(f"웹 문서 로딩: {len(web_docs)}개")
print(f"내용 미리보기:\n{web_docs[0].page_content[:300]}...")


## **4. 텍스트 분할 (Text Splitting)**

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter: 단락 → 줄 → 문장 → 단어 순서로 분할 시도
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,      # 청크 최대 크기 (문자 수)
    chunk_overlap=30,    # 청크 간 중첩 (문맥 보존)
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(documents)

print(f"분할 전: {len(documents)}개 문서")
print(f"분할 후: {len(chunks)}개 청크")
print()
print("=== 청크 예시 ===")
for i, chunk in enumerate(chunks[:3]):
    print(f"청크 {i+1} (길이: {len(chunk.page_content)}자):")
    print(f"  {chunk.page_content}")
    print(f"  메타데이터: {chunk.metadata}")
    print()


**청크 크기 선택 가이드:**
- `chunk_size`: 너무 작으면 문맥 손실, 너무 크면 노이즈 증가 (보통 200~1000)
- `chunk_overlap`: 청크 경계에서 정보 누락 방지 (chunk_size의 10~20%)


## **5. 임베딩 & 벡터 스토어**

**5-1. 임베딩 생성**

In [ ]:
from langchain_openai import OpenAIEmbeddings

# 텍스트를 고차원 벡터로 변환
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 단일 텍스트 임베딩 테스트
sample_text = "이노코어의 주요 제품은 무엇인가요?"
vector = embeddings.embed_query(sample_text)

print(f"텍스트: {sample_text}")
print(f"벡터 차원: {len(vector)}")
print(f"벡터 앞 5개 값: {vector[:5]}")


**5-2. FAISS 벡터 스토어 생성**

In [ ]:
from langchain_community.vectorstores import FAISS

# 청크를 임베딩하여 벡터 스토어에 저장
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f"벡터 스토어 생성 완료!")
print(f"저장된 벡터 수: {vectorstore.index.ntotal}")

# 로컬 저장 (선택)
vectorstore.save_local("/tmp/faiss_index")
print("벡터 스토어 저장 완료: /tmp/faiss_index")


## **6. 검색 (Retrieval)**

**6-1. 유사도 검색**

In [ ]:
# 직접 유사도 검색
query = "이노코어 회사 소개"
results = vectorstore.similarity_search(query, k=2)

print(f"쿼리: {query}")
print()
for i, doc in enumerate(results, 1):
    print(f"[결과 {i}] 출처: {doc.metadata['source']}")
    print(f"내용: {doc.page_content}")
    print()


**6-2. 점수 포함 검색**

In [ ]:
# 유사도 점수와 함께 검색
query = "RAG 구성 요소"
results_with_scores = vectorstore.similarity_search_with_score(query, k=3)

print(f"쿼리: {query}")
print()
for doc, score in results_with_scores:
    print(f"유사도 점수: {score:.4f} (낮을수록 유사)")
    print(f"출처: {doc.metadata['source']}")
    print(f"내용: {doc.page_content[:100]}...")
    print()


**6-3. Retriever 객체 생성**

In [ ]:
# LangChain 체인에서 사용할 Retriever 생성
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# 검색 테스트
docs = retriever.invoke("LLM의 한계점")
print(f"검색 결과: {len(docs)}개 문서")
for doc in docs:
    print(f"  - {doc.page_content[:80]}...")


## **7. RAG 체인 구성**

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# RAG 프롬프트 템플릿
prompt = ChatPromptTemplate.from_template("""
당신은 도움이 되는 AI 어시스턴트입니다.
아래 컨텍스트를 바탕으로 질문에 답하세요.
컨텍스트에 없는 내용은 '주어진 정보에는 없습니다'라고 답하세요.

컨텍스트:
{context}

질문: {question}

답변:
""")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# RAG 체인 구성
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 질문 테스트
questions = [
    "이노코어는 어떤 회사인가요?",
    "PRISM-AI 플랫폼의 기능은 무엇인가요?",
    "워크숍 3일차에는 무엇을 배우나요?",
]

for q in questions:
    print(f"Q: {q}")
    answer = rag_chain.invoke(q)
    print(f"A: {answer}")
    print("-" * 60)


## **8. 대화형 RAG (Conversational RAG)**

대화 히스토리를 반영하여 이전 대화 맥락을 고려한 RAG를 구현합니다.

```
대화 히스토리 + 새 질문
         │
         ▼
[질문 재구성] ← history-aware retriever
         │
         ▼
[관련 문서 검색]
         │
         ▼
[대화 히스토리 + 문서 + 질문] → LLM → 답변
```


In [ ]:
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 1단계: 히스토리를 고려한 질문 재구성 프롬프트
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", """대화 히스토리와 새 질문을 보고,
대화 히스토리 없이도 이해할 수 있는 독립적인 질문으로 재구성하세요.
질문에 답하지 말고, 필요한 경우에만 재구성하세요."""),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_prompt
)

# 2단계: QA 체인 프롬프트
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 도움이 되는 AI 어시스턴트입니다.
아래 컨텍스트를 바탕으로 질문에 답하세요.
컨텍스트에 없는 내용은 '주어진 정보에는 없습니다'라고 답하세요.

컨텍스트:
{context}"""),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# 최종 RAG 체인 조합
rag_chain_with_history = create_retrieval_chain(history_aware_retriever, question_answer_chain)


In [ ]:
# 대화형 RAG 실행
chat_history = []

conversations = [
    "이노코어에 대해 알려주세요.",
    "그 회사의 주요 제품은 무엇인가요?",    # '그 회사'가 이노코어를 참조
    "그것의 처리 데이터 규모는 어떻게 되나요?",  # '그것'이 PRISM-AI를 참조
]

for question in conversations:
    result = rag_chain_with_history.invoke({
        "input": question,
        "chat_history": chat_history
    })

    answer = result["answer"]
    chat_history.extend([
        HumanMessage(content=question),
        AIMessage(content=answer)
    ])

    print(f"Q: {question}")
    print(f"A: {answer}")
    print("-" * 60)

print(f"\n총 대화 턴: {len(chat_history)//2}회")


## **정리**

| 단계 | 구성요소 | 역할 |
|------|----------|------|
| 오프라인 | Document Loader | 소스에서 문서 로딩 |
| 오프라인 | Text Splitter | 문서를 청크로 분할 |
| 오프라인 | Embedding + Vector Store | 청크를 벡터화하여 저장 |
| 온라인 | Retriever | 질문과 유사한 청크 검색 |
| 온라인 | LLM | 검색 결과 기반 답변 생성 |

**RAG 성능 향상 팁:**
- **청크 크기 최적화**: 도메인에 따라 적절한 크기 조정
- **Reranking**: 검색 결과를 재순위화하여 정확도 향상
- **Hybrid Search**: 키워드 검색 + 시맨틱 검색 병행
- **Self-Query**: LLM으로 메타데이터 필터 자동 생성
